# 백엔드와 데이터베이스 - 데이터가 남는 서비스 만들기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 여는 이야기 - 새로고침해도 남는 데이터

### 상품 목록 화면에서 데이터를 바꾸면 새로고침 뒤에도 남을까요?

목업 데이터(화면을 확인하기 위한 예시 데이터)로 만든 상품 목록 화면을 떠올려 봐요. 저장 기능 없이 화면의 메모리(실행 중 잠깐 쓰는 기억 공간)에만 변경 내용을 담았다면 새로고침할 때 사라져요.

저장 원리를 익히기 위해 오늘은 작은 Todo(할 일) 앱으로 연습해요. 서비스가 되려면 데이터가 **남아 있어야** 해요. **데이터베이스(DB)** 는 데이터를 저장하고 필요한 것을 찾아 꺼낼 수 있게 하는 저장소예요. 오늘은 화면 뒤에서 요청을 처리하는 **백엔드** 와 데이터베이스를 연결해 봐요.

![하드디스크 내부 - 원반(플래터)과 자기 헤드](https://commons.wikimedia.org/wiki/Special:FilePath/Hard_disk_platters_and_head.jpg?width=640)
*우리 데이터는 결국 이런 물리적 장치 어딘가에 실제로 적힙니다.*

### 오늘의 핵심 한 문장

> **백엔드는 요청을 처리하고, 데이터베이스는 그 결과를 남깁니다. 코드는 AI가 쓰지만, 무엇을 어떤 모양으로 저장하고 어떻게 안전하게 지킬지는 여러분이 정합니다.**

오늘의 결과물은 **데이터 설계도 한 장과 새로고침해도 할 일이 남는 Todo(할 일) 앱** 입니다. 설계도에는 저장할 표·항목·관계를 적고, 앱은 내 컴퓨터의 DB와 Supabase(클라우드 DB 서비스)에 각각 연결해 확인합니다. 클라우드는 인터넷으로 다른 컴퓨터의 저장 공간이나 기능을 빌려 쓰는 방식입니다.

## 백엔드가 실제로 하는 일 - 요청을 받아 처리하고 돌려주기

## 그래서 서버 안에선 무슨 일이? - 요청 받기 → 처리 → 응답

요청이 서버에 도착한 뒤 백엔드가 하는 일은 한 문장으로 정리할 수 있어요. **요청을 받아서, 처리하고, 응답을 돌려줘요.** 서버는 브라우저의 요청을 받아 응답을 보내는 컴퓨터나 프로그램입니다. 프론트(브라우저)가 "이 상품 목록 줘", "이 주문 접수해 줘"라고 요청을 보내면, 백엔드는 그걸 해석하고 → 규칙을 검사하고 계산하고 데이터를 읽거나 쓰고 → 결과를 응답으로 돌려줍니다. (출처: MDN, "Client-Server overview")

목업 데이터로 만든 화면이 '창고의 완성품을 그대로 꺼내 주는 것'이라면, 백엔드는 **주문마다 그때그때 만들어 주는 동적 처리** 입니다. 백엔드를 식당의 주방에 비유하면 -

- 홀(프론트)에서 **주문서(요청)** 를 받고
- **레시피(서비스의 처리 규칙)** 대로 조리하고
- **창고(데이터베이스)** 에서 재료를 꺼내고
- 완성한 요리를 **내보냅니다(응답).**

손님은 주방 안을 보지 못하고, 오직 '주문'과 '나온 요리'만 주고받습니다 - 이게 백엔드가 자신을 바깥에 감추는 방식입니다.
지금까지 말로 본 '요청 → 처리 → 응답'이 실제 코드에서는 어떤 모습인지, Python의 Flask 예시로 구경만 해 봐요. 아래는 읽기용 예시라 이 노트북에서는 실행할 수 없어요.

```python
from flask import Flask
app = Flask(__name__)

@app.route("/todos")             # 요청: 이 주소가 요청을 받는 창구예요
def todos():                     # 처리: 요청이 오면 이 함수가 실행돼요
    items = ["할 일 추가하기", "완료 표시하기"]
    return {"todos": items}      # 응답: 결과를 돌려줘요
```

문법은 몰라도 괜찮고, 핵심은 **요청이 오면 내가 미리 둔 코드가 실행된다** 는 거예요. 실행은 백엔드(서버 쪽)에서 일어나고, 오늘 실습에서는 에이전트가 이 코드를 대신 짜요.

### 개념 정리: 백엔드가 열어 주는 '창구' - 엔드포인트와 API

백엔드는 통째로 열려 있지 않고, 정해진 몇 개의 **창구(엔드포인트)** 만 프론트에 열어 줍니다. 창구 하나는 **"주소(URL) + 동작"** 의 조합이고, 그게 곧 하나의 기능입니다.

- `GET /products` → "상품 목록 주세요"
- `POST /login` → "이 정보로 로그인 처리해 주세요"

이렇게 백엔드가 밖에 공개한 창구들의 묶음을 **API** 라고 부릅니다. REST(자원을 주소로 구분하고 정해진 동작으로 다루는 API 설계 방식)의 핵심은 하나 - 주소는 '무엇'(명사), 동작은 메서드(요청의 동작)로. 그래서 `/getProducts`(동사)가 아니라 `/products`(명사)로 쓰고, 가져올지·만들지·지울지는 동작이 정합니다. (출처: Microsoft Learn, "Web API design best practices")

> 처음 배우는 여러분에게 필요한 건 완벽한 설계가 아니라 **"내 서비스엔 어떤 창구가 필요하지?"** 를 그려 보는 판단입니다. 쇼핑몰이라면 '상품 목록·상품 상세·장바구니 담기·주문하기' 정도. AI에게 맡기더라도, 필요한 창구를 스스로 말할 수 있으면 훨씬 정확한 결과가 나옵니다.

### 개념 정리: 거의 모든 서비스가 하는 네 가지 - CRUD

대부분의 서비스가 데이터에 대해 하는 일은 사실상 넷뿐입니다 - **만들기(Create)·읽기(Read)·고치기(Update)·지우기(Delete).** 이 네 글자를 묶어 **CRUD** 라고 부릅니다. (출처: MDN Glossary, "CRUD")

인스타그램의 '게시물 올리기·보기·수정·삭제', 쇼핑몰의 '상품 등록·조회·수정·삭제'처럼 - 겉모습만 다를 뿐 속은 전부 이 네 동작의 반복입니다. 앞의 API 창구와도 짝이 맞아서(만들기=POST, 읽기=GET…), **"어떤 창구가 필요할까?"** 는 대개 **"이 데이터에 CRUD 중 무엇을 허용할까?"** 로 바꿔 생각할 수 있습니다.

> 여기선 'API가 하는 일' 수준까지만. 실제로 데이터를 저장·조회하는 부분은 오늘 이어서 살펴봅니다.

### 개념 정리: 프론트와 백엔드의 공용어 - JSON

프론트와 백엔드는 서로 다른 환경에서 돌아가는데, 데이터를 주고받을 땐 **JSON** 이라는 공통 형식을 씁니다. '키: 값' 쌍으로 적는 가벼운 텍스트죠.

```json
{ "name": "홍길동", "age": 30, "isMember": true }
```

JSON이 사실상 표준이 된 이유는 - **사람이 읽기 쉬운 텍스트인 데다 언어를 가리지 않기** 때문입니다. 어떤 프론트와 어떤 백엔드를 붙여도 서로 알아듣습니다. (출처: MDN Glossary, "JSON"; RFC 8259)

> JSON은 '키-값' 쌍으로 데이터를 표현해요. 문법을 외울 필요는 없고 **"아, 이렇게 생긴 걸로 데이터를 주고받는구나"** 정도면 충분합니다.

### 개념 정리: 백엔드는 무슨 언어로 짜나

백엔드도 언어와 '프레임워크(개발 도구)'를 고릅니다. 대표 조합만 -

- **JavaScript/TypeScript** → Node.js 위에서 **Express**(가볍고 기본적)·NestJS(구조가 잡힌)
- **Python** → **Django**(다 들어간 종합형)·Flask(가벼움)·**FastAPI**(빠르고 요즘 뜨는)
- 그 밖에 Ruby on Rails, Go, Java/Spring 등

처음 만드는 서비스라면 무난한 기본값은 둘 중 하나입니다. **(1) JavaScript 풀스택** - 프론트도 백엔드도 JS로 통일하면 언어 하나만 익혀도 되고 자료가 많습니다. **(2) Python** - 문법이 읽기 쉽고 AI 기능을 붙이기 좋습니다. (참고: 언어·프레임워크 인기도는 Stack Overflow 2025 개발자 설문 기준)

> 어느 쪽도 '정답'은 아니고, 어차피 **코드는 AI가 짭니다.** 여러분은 개념(요청·창구·CRUD·검증)만 잡고 방향을 판단하면 됩니다. "이 프레임워크가 제일 빠르다/최고다" 같은 말에 휘둘릴 필요 없습니다.

### 보안 이야기: 백엔드의 진짜 가치 - "프론트는 못 믿는다"

백엔드의 진짜 가치는 데이터를 나르는 게 아니라 **규칙을 지키는 것** 입니다 - 재고가 남았는지, 할인 적용한 최종 가격이 얼마인지, 이 사용자가 이 글을 지울 권한이 있는지. 이런 판단을 **비즈니스 로직** 이라고 합니다.

여기서 가장 중요한 보안 원칙 하나 - **가격·수량·권한 같은 검증은 반드시 백엔드에서 다시 해야 합니다.** 프론트(브라우저)는 사용자의 컴퓨터에서 돌아가므로, 사용자가 얼마든지 조작할 수 있기 때문입니다. (출처: OWASP Input Validation Cheat Sheet; 권한 통제는 OWASP Top 10 A01 Broken Access Control)

> 예를 들어 화면에 "가격 10,000원"이라고 떠 있어도, 악의적 사용자는 브라우저에서 그 값을 **100원으로 바꿔** 주문 요청을 보낼 수 있습니다. 그래서 **최종 가격은 반드시 서버가 자기 데이터로 다시 계산** 해야 하고, 프론트가 보낸 값을 그대로 믿으면 안 됩니다. 프론트의 검증은 사용자 편의(빠른 오류 안내)를 위한 것이고, **진짜 방어선은 언제나 백엔드** 입니다.

주방이 주문을 처리해도 재료를 둘 창고가 필요하듯, 백엔드가 요청을 처리한 결과도 남겨 둘 곳이 필요합니다. 결과를 화면의 메모리에만 두면 새로고침할 때 사라지므로, 이제 데이터베이스에 저장하는 방법을 살펴봅니다.

## 데이터를 저장할 곳이 필요한 이유

### 데이터베이스가 뭐냐면 - 도서관, 그리고 마법 장난감 상자

두 가지 비유로 살펴보겠습니다.

- **도서관** - 도서관에는 책이 수만 권 있지만, 원하는 책을 금방 찾습니다. **데이터베이스 = 도서관, 데이터 = 책, 그걸 관리하는 소프트웨어 = 사서** 입니다.
- **마법 장난감 상자** - 장난감을 마구 넣어도, "파란색 자동차만 다 꺼내 줘"라고 하면 정리해서 딱 그것만 꺼내 줍니다.

핵심은 이겁니다 - 데이터베이스는 그냥 쌓아 두는 곳이 아니라, **원하는 것만 빠르게 찾아 주는** 곳입니다. (출처: Stackby, "What is a database: A beginner's guide")

![미국 의회도서관 열람실](https://commons.wikimedia.org/wiki/Special:FilePath/LOC_Main_Reading_Room_Highsmith.jpg?width=640)
*데이터베이스는 이렇게 '잘 정리돼 언제든 원하는 걸 찾을 수 있는 서고'에 가깝습니다. (이미지: Wikimedia Commons)*

### 그럼 엑셀·메모장이랑 뭐가 다른가요?

좋은 질문입니다. 작을 때는 엑셀로도 됩니다. 하지만 데이터베이스는 세 가지가 다릅니다.

- **규칙(스키마)이 있습니다** - "이 표에는 이름·가격·분류가 들어간다"처럼 모양을 정해 둡니다. 그래서 뒤죽박죽이 안 됩니다. 엑셀과 달리, 대부분의 데이터베이스는 이런 규칙을 강제해 데이터가 뒤죽박죽이 되는 걸 막아 줍니다. (출처: Servermall, "Simple Database Basics")
- **수만 건에서도 순식간에 찾습니다** - 엑셀은 커지면 느려지고 멈추지만, DB는 그러라고 만들어졌습니다.
- **여러 사람이 동시에 안전하게** 쓸 수 있습니다 - 서비스에는 필수죠.

> 오늘도 마찬가지입니다. DB에 넣고 꺼내는 코드는 **AI가 씁니다.** 여러분은 "무엇을 어떤 모양으로 저장할지"와 "어떤 DB를 고를지"를 판단하면 됩니다.

### 스케일 이야기: 세상에서 가장 큰 데이터베이스는 얼마나 클까

"데이터베이스가 크다"는 게 얼마나 큰 건지 감이 안 올 수 있습니다. 놀라운 사례 몇 가지.

- **미국 의회도서관** - 세계에서 가장 큰 도서관으로, 책·사진·지도·녹음·필름 등 **1억 7,800만 점 이상** 을 소장합니다(책장을 늘어놓으면 약 838마일). 매일 약 1만 5천 점을 받아 1만여 점을 새로 소장하고, 디지털로 보존하는 자료만 해도 **페타바이트(PB) 단위** 에 이릅니다. (출처: Library of Congress, "Fascinating Facts", loc.gov)
- **CERN(유럽입자물리연구소)** - LHC 검출기에서는 **초당 약 10억 번** 의 입자 충돌이 일어나, 이론상 초당 **1페타바이트** 의 데이터가 쏟아집니다. 하지만 '트리거'가 **의미 있는 극소수(1%도 안 됨)만 골라** 저장하죠. 그렇게 걸러 저장해도 CERN의 데이터 저장 용량은 **2023년에 1엑사바이트(=100만 테라바이트)를 넘어섰습니다.** 큰 데이터에서는 **무엇을 저장할지 고르는 일** 이 저장장치 크기만큼 중요하다는 걸 보여 줍니다. (출처: CERN, home.cern; The Register, 2023)

그런데 재미있는 사실 - 예전엔 "얼마나 크냐"로 자랑했지만, 요즘은 **클라우드** 덕분에 처음 시작해도 이런 거대한 기술을 **무료로 조금 빌려** 씁니다. 규모의 문제가 아니라, 이제는 **누구나 쓸 수 있는** 도구가 된 것이죠.

![데이터베이스가 앱의 요청과 영구 저장소를 연결하는 그림](https://ebhtmardjzsiyxjncvbb.supabase.co/storage/v1/object/public/images/01-vibe-coding-web/day08_infographic_dbrole.png)
*요청을 처리하는 서비스와, 껐다 켜도 남는 저장소의 관계*

## 왜 종류가 많을까 - 데이터베이스의 역사

## 파일 상자에서 클라우드까지 - 데이터베이스가 걸어온 길

---

### 반복을 줄이려는 60년의 이야기

쇼핑몰 주문 목록을 엑셀에 적던 두 사람이 같은 행을 동시에 고쳤습니다. 한 사람의 수정이 다른 사람의 내용을 덮어쓰면서, 하루 주문이 꼬였습니다. 데이터를 함께 쓰고 바뀐 내용을 잃지 않으려는 문제가 데이터베이스를 낳았습니다.

"데이터베이스도 종류가 왜 이렇게 많아?" - 프레임워크와 똑같습니다. 매 시대 **병목이 달랐고**, 그때마다 새 도구가 나왔습니다.

- **1960년대 - 그냥 파일에 담다.** 처음엔 데이터를 파일에 쭉 적어 뒀습니다. 양이 늘자 원하는 걸 찾기가 너무 번거로워졌습니다.
- **1970년 - "표로 나누자" (관계형의 탄생).** IBM의 연구원 **에드거 코드(Edgar Codd)** 가 "데이터를 **표(테이블)** 로 나누고, 표끼리 **관계** 로 잇자"는 논문을 냈습니다. 같은 정보를 여기저기 중복해 적지 않아도 되게 한 혁신이었습니다. (출처: Codd, "A Relational Model of Data for Large Shared Data Banks", 1970)
- **1974년 - 질의 언어 등장.** 이 표들에게 말을 거는 언어 **SEQUEL** 이 만들어졌습니다(뒤에 **SQL** 로 개명). 50여 년이 지난 지금도 거의 그대로 쓰입니다 - 잘 만들어졌다는 뜻이죠.
- **1979년 - 첫 상용 관계형 DB(Oracle).** 회사들이 쓰기 시작했습니다.
- **2000년대 - 인터넷 폭발, 그리고 NoSQL.** 데이터가 폭증하고 형태도 제각각이 되자, 엄격한 표만으로는 버거워졌습니다. 그래서 더 유연하고 대규모에 강한 **NoSQL** 이 등장했습니다. (출처: Dataversity, "A Brief History of Non-Relational Databases")
- **오늘날 - 클라우드 데이터베이스.** 이제는 켜 두고 관리하는 일까지 클라우드가 대신합니다. 처음 시작해도 무료로 진짜 DB를 씁니다.

이름이 많은 건 어렵게 만들려는 게 아니라, **각 시대의 문제를 풀며 쌓여 온 것** 입니다. (출처: Cockroach Labs, "A brief history of databases")

![천공카드가 가득 쌓인 옛 자료 보관실](https://commons.wikimedia.org/wiki/Special:FilePath/IBM_card_storage.NARA.jpg?width=640)
*'파일 상자' 시절 - 데이터는 이렇게 천공카드로 물리적으로 쌓였습니다. 찾는 것도, 중복되지 않게 두는 것도 큰일이었죠. (이미지: Wikimedia Commons)*

### 이야기 하나: 코드의 '표 혁명'과 IBM의 망설임

앞의 연표에서 1970년에 **에드거 코드(Edgar Codd)** 가 등장했죠. 이 사람 이야기는 조금 더 들을 만합니다.

1970년 당시 데이터는 나뭇가지처럼 얽힌 복잡한 구조로 저장됐습니다. 원하는 걸 꺼내려면 그 가지를 타고 들어가는 전용 프로그램을 매번 짜야 했죠. 코드는 IBM 연구소에서 이걸 지켜보다 **"그냥 엑셀 같은 표로 두고, 표끼리 관계로 이으면 안 되나?"** 라는 논문을 냈습니다(*Communications of the ACM*, 1970). 데이터의 '생김새'와 '저장 방식'을 분리하자는, 지금 보면 당연하지만 당시엔 파격적인 생각이었습니다.

재미있는 반전은 **정작 그 IBM이 처음엔 미지근했다** 는 겁니다. 이미 IMS라는 기존 데이터베이스 제품이 잘 팔리고 있었거든요. 새 아이디어가 자기네 효자 상품을 갉아먹을까 봐 망설인 거죠. 결국 IBM도 뒤늦게 'System R'이라는 연구를 시작하지만, 이 망설임의 틈을 노린 회사가 따로 있었습니다. (출처: Wikipedia, "Edgar F. Codd"; Codd, 1970)

> 새 기술이 좋아도, 기존에 잘 팔리는 걸 가진 쪽은 주저하게 됩니다 - 기술의 역사에서 계속 반복되는 장면입니다.

### 이야기 둘: 'SQL'은 원래 '시퀄'이었다

표에게 말을 거는 언어 SQL에도 사연이 있습니다. IBM의 System R 팀(도널드 체임벌린, 레이먼드 보이스)이 처음 만든 이름은 **SEQUEL** - "Structured English Query Language(구조화된 영어식 질의 언어)"의 약자였습니다. 이름 그대로 **영어 문장처럼 읽히게** 만들자는 뜻이었죠.

그런데 'SEQUEL'이 이미 어느 영국 항공기 회사의 **상표** 로 등록돼 있었습니다. 변호사 편지를 받은 팀은 모음을 빼고 **SQL** 로 줄였습니다. 오늘날 사람마다 이걸 "에스큐엘"이라 읽기도 하고 "시퀄"이라 읽기도 하는 이유가 여기 있습니다 - 만든 사람들조차 옛 이름 그대로 **"시퀄"** 이라 부르거든요. (출처: Wikipedia, "SQL"; 체임벌린 회고)

> 그러니 어느 발음을 들어도 당황하지 마세요. 둘 다 같은 걸 가리키고, 둘 다 맞습니다.

### 이야기 셋: 관계형을 만든 IBM보다 빨랐던 신생 회사

IBM이 망설이는 사이, 코드의 논문을 읽고 **"이거 돈이 되겠다"** 고 확신한 사람이 있었습니다. 바로 **래리 엘리슨(Larry Ellison)** 입니다.

그는 1977년 동료들과 작은 회사를 차렸습니다. 이 회사는 이름을 여러 번 바꾸는데(SDL → 관계형소프트웨어 → 결국 **오라클(Oracle)**), 1979년 **첫 상용 관계형 데이터베이스** 를 시장에 내놓습니다. 관계형이라는 아이디어를 **처음 만든 IBM(SQL/DS는 1981년)보다 2년이나 앞선** 것이었죠. 거대 기업이 자기 논문을 붙들고 고민하는 동안, 신생 스타트업이 그 아이디어로 먼저 시장을 차지한 겁니다.

> 재미난 뒷이야기 - 첫 제품 버전을 '1'이 아니라 '2'라고 붙였습니다. 갓 나온 소프트웨어를 아무도 안 믿을까 봐, 처음부터 "이미 한 번 개선했다"는 인상을 주려던 것이었죠. (출처: Wikipedia, "Oracle Corporation" / "Oracle Database")

### 이야기 넷: NoSQL의 진짜 뜻은 'No'가 아니라 'Not Only'

시간이 흘러 2000년대, 구글과 아마존은 전에 없던 고민에 빠집니다 - **전 세계가 동시에 접속하는 규모** 의 데이터를 어떻게 감당하지? 기존의 반듯한 표 방식만으론 버거웠습니다. 그래서 각자 새로운 방식을 논문으로 공개하죠(구글 'Bigtable' 2006, 아마존 'Dynamo' 2007). 표의 엄격함을 조금 포기하는 대신, **엄청난 규모와 속도** 를 얻는 방식이었습니다.

이 흐름에 이름이 붙은 건 2009년, 샌프란시스코의 한 개발자 모임에서였습니다. 짧고 검색하기 좋은 이름을 찾다가 **"NoSQL"** 로 정해졌죠. 그런데 이 이름이 오해를 낳았습니다 - "SQL을 쓰지 말자"는 뜻으로 들렸거든요. 그래서 나중에 **"Not Only SQL(SQL만 쓰는 게 아니다)"** 로 다시 해석됩니다. 실제로도 NoSQL은 관계형을 대체한 게 아니라, **용도에 따라 나눠 쓰는** 또 하나의 선택지가 되었습니다. (출처: Wikipedia, "NoSQL"; Martin Fowler, "NosqlDefinition")

> 그래서 오늘날 큰 서비스들은 "SQL이냐 NoSQL이냐"로 싸우지 않습니다. **둘 다** 필요한 곳에 씁니다.

### 개념 정리: 표냐, 유연한 상자냐 - SQL과 NoSQL

오늘날 DB는 크게 두 갈래입니다. 문법이 아니라 **성격** 만 알면 됩니다.

| | **SQL** | **NoSQL** |
|---|---|---|
| 생김새 | 엑셀 같은 **표** | 유연한 **문서/덩어리** (모양 자유) |
| 규칙(스키마) | 엄격하게 정해 둠 | 느슨하거나 없음 |
| 강한 곳 | 관계가 분명한 데이터, **정확성**(돈·주문) | 대규모·빠르게 변하는 데이터 |
| 대신 포기 | 유연함이 덜함 | 표끼리 잇기(조인)·엄격한 거래 일부 |

정답은 없습니다. 넷플릭스·우버 같은 큰 서비스도 **둘 다** 씁니다(용도별로 골라 쓰는 것을 '폴리글랏'이라 부릅니다). 다만 **처음 만드는 서비스라면 SQL이 무난한 기본값** 입니다 - 자료가 많고, 관계가 분명한 데이터를 다루기 좋기 때문입니다. (출처: Coursera, "SQL vs. NoSQL"; MongoDB, "NoSQL vs SQL")

## SQL 구경과 데이터의 모양

## SQL은 거의 영어입니다 - 알아보는 정도면 충분

---

### SQL 한 줄 구경하기

카페 사장님이 "케이크 상품의 이름과 가격만 보고 싶어요"라고 했는데, 상품이 수백 개라 눈으로 찾기 어려워요. 필요한 조건의 데이터만 골라 달라고 DB에 요청하는 말이 필요해요. 그 요청을 쓰는 언어가 SQL이에요.

**SQL** 은 데이터베이스에게 말을 거는 언어입니다. 영어 단어로 명령을 적습니다. 한 줄만 보겠습니다.

```sql
SELECT 이름, 가격 FROM 상품 WHERE 분류 = '케이크';
```

풀어 읽으면 이렇습니다 - **"상품 표에서(FROM 상품) 분류가 '케이크'인 것들의(WHERE) 이름과 가격을 줘(SELECT)."** 정말 영어 문장에 가깝죠.

넣고·고치고·지우는 것도 단어만 바뀝니다.

- `SELECT` - 꺼내 보기 (조회)
- `INSERT` - 새로 넣기
- `UPDATE` - 고치기
- `DELETE` - 지우기

앞서 본 **CRUD**(만들기·읽기·고치기·지우기)가 SQL에서는 각각 `INSERT`·`SELECT`·`UPDATE`·`DELETE`에 대응해요.

> **이걸 외울 필요는 없습니다.** 실제 SQL은 AI가 써 줍니다. 여러분은 SQL 문장을 보고 **"아, 이 표에서 뭘 꺼내는(넣는) 거구나"를 알아보는 정도** 면 충분합니다.

### 표로 생각하기 - 행, 열, 그리고 관계

관계형 데이터베이스는 결국 **표** 입니다. 엑셀을 떠올리면 됩니다.

- **표 하나 = 한 종류** (예: `상품` 표)
- **행(row) = 한 건** (예: 딸기 케이크 하나)
- **열(column) = 항목** (예: 이름, 가격, 분류)

그리고 표끼리 **관계** 로 잇습니다. 예를 들어 `주문` 표는 "어떤 상품을 샀는지"를 `상품` 표를 **가리키는** 방식으로 적습니다 - 정확히는 각 상품에 붙은 **고유 번호(ID)** 로 가리킵니다. 상품 정보를 주문마다 통째로 베끼지 않아도 되죠. 이 "표로 나누고 관계로 잇기"가 바로 **관계형(relational)** 이라는 이름의 뜻입니다.

### 해 보기: 쇼핑몰 데이터를 '표 몇 개'로 쪼개면

튜토리얼로 만든 쇼핑몰을 예로, 데이터를 어떻게 표로 나누는지 감을 잡아 봅시다. 문법은 필요 없고 **'무엇을 따로 둘까'** 만 생각하면 됩니다.

한 덩어리로 뭉치면 이렇게 됩니다 - *"홍길동이 3월 2일에 초코케이크 2개를 12,000원에 샀고, 홍길동의 주소는 …"*. 주문이 100건이면 홍길동 주소가 100번 적힙니다. 그래서 **성격이 다른 것끼리 표를 나눕니다.**

| 표 | 담는 것 | 예시 한 줄 |
|---|---|---|
| **회원** | 사람 정보 | 1번 · 홍길동 · 서울… |
| **상품** | 파는 것 | 7번 · 초코케이크 · 12,000원 |
| **주문** | 언제 누가 무엇을 | 55번 · **1번 회원** · **7번 상품** · 2개 · 3/2 |

핵심은 주문 표가 이름·주소를 **다시 적지 않고** "1번 회원"이라고만 가리키는 것입니다(이걸 뒤에서 '정규화'라 부릅니다). 주소가 바뀌면 회원 표 한 곳만 고치면 끝이죠. 표를 잇는 이 '번호'를 **키(key)** 라고 부릅니다 - 표와 표를 잇는 고리입니다.

> 이걸 직접 설계할 필요는 없습니다. AI에게 **"쇼핑몰에 필요한 표들과 그 관계를 그려줘"** 라고 하면 위 같은 그림을 만들어 줍니다. 여러분이 할 일은 그 그림을 읽고 **"내 서비스엔 어떤 표가 더 필요하지?"** 를 판단하는 것입니다.

### 표는 어떻게 '이어지나' - 1:N과 N:M

표를 여러 개로 나눴으면, 이제 서로 **어떻게 이어지는지** 를 볼 차례입니다. 관계는 대부분 두 가지 모양입니다.

**① 일대다(1:N) - 한쪽이 여러 개를 가진다**
회원 1명은 주문을 여러 건 남기지만, 주문 1건은 **한 회원** 의 것입니다. 이럴 땐 '주문' 표에 "누구의 주문인지"를 가리키는 칸(회원 번호) 하나만 넣으면 끝입니다. 블로그의 '글 1개 ↔ 댓글 여러 개', 유튜브의 '채널 1개 ↔ 영상 여러 개'가 모두 같은 모양이죠.

**② 다대다(N:M) - 양쪽 다 여러 개를 가진다**
학생 1명은 여러 수업을 듣고, 수업 1개도 여러 학생이 듣습니다. 이건 양쪽 표 어디에도 상대를 다 적을 수 없습니다(한 칸에 여러 개를 욱여넣으면 검색이 깨집니다). 그래서 **'누가 어떤 수업을 듣는다'는 연결 표**(수강신청 표)를 하나 새로 만들어, 학생 번호와 수업 번호를 짝지어 한 줄씩 적습니다. 그러면 N:M이 **'연결 표를 사이에 둔 두 개의 1:N'** 으로 깔끔하게 풀립니다.

> 판단 감각 하나면 됩니다 - **"A 하나가 B 여럿을, 동시에 B 하나도 A 여럿을 갖나?"** 그렇다면 N:M이라 중간 표가 필요하고, 한쪽만 여러 개면 1:N입니다. 이 판단만 해 주면, 표를 만드는 SQL은 AI가 씁니다.

### 작동 원리: DB는 왜 그렇게 빠를까 - 책 뒤 '찾아보기(색인)'

수만 권 중에서 원하는 책을 도서관이 금방 찾는 비결이 뭘까요? 데이터베이스도 같은 트릭을 씁니다 - **색인(index)** 입니다.

두꺼운 전공책 맨 뒤에는 '찾아보기'가 있죠. "삼투압 → 245쪽"처럼요. 그 덕에 처음부터 한 장씩 넘기지 않고 **바로** 원하는 곳으로 갑니다. DB의 색인이 정확히 이겁니다 - 자주 찾는 항목(예: 이메일, 상품명)에 색인을 만들어 두면, 수백만 건에서도 **순식간에** 찾습니다. 색인이 없으면? 처음부터 전부 뒤집니다(그래서 느려지죠).

> 여러분이 외울 건 없습니다. 서비스가 커져서 "검색이 느려요" 싶을 때, AI에게 **"자주 검색하는 항목에 색인을 넣어줘"** 라고 하면 됩니다. 공짜는 아니어서(넣고 빼는 게 살짝 느려짐) 아무 데나 다 걸진 않지만, 개념만 알아 두면 "왜 느리지?"의 흔한 답이 됩니다.

### 조금 더: 인덱스는 왜 빨라지나 - 그리고 공짜가 아니다

색인(인덱스)은 '값 → 행 위치'를 미리 정리해 둔 구조라, 수백만 행에서도 몇 단계만에 원하는 행에 닿습니다(그래서 빠릅니다). 대신 **공짜가 아닙니다** - 넣고 고칠 때마다 색인도 갱신해야 하고 공간도 듭니다. 그래서 **많이 거는 게 아니라, 실제로 자주 쓰는 검색·정렬에 맞게 최소한만** 거는 게 빠른 DB입니다.

> 활용 예 - AI에게 "색인 넣어줘"만 하지 말고, 먼저 **"사용자는 자기 주문만 최근순 20개 본다"** 처럼 실제 화면의 질문을 문장으로 주세요. 그럼 AI가 맞는 색인을 제안합니다.

<details class="lms-extra">

<summary><b>더 깊이</b> (개발자로 더 갈 사람만 - 건너뛰어도 됩니다)</summary>

많은 관계형 DB의 기본 색인은 B-트리 계열입니다. 한 노드에 수십～수백 갈래를 둬서, 세 층만 내려가도 수백만 범위를 가릅니다(그래서 'O(log n)' - 행이 10배로 늘어도 단계는 조금만 늘어남). DB는 행 하나씩이 아니라 '페이지' 단위로 읽으므로, 실제 속도는 페이지 수·캐시 적중에도 달렸습니다. `(user_id, created_at)` 같은 **복합 인덱스** 는 순서가 중요하고, `WHERE name LIKE '%민수'`(앞 와일드카드)나 `WHERE LOWER(email)=...`(열에 함수를 씌움)처럼 색인을 못 타는 경우도 있습니다. 그래서 AI가 제안한 SQL은 실제와 비슷한 데이터로 `EXPLAIN`을 돌려 어느 색인을 탔는지 확인하는 게 안전합니다. 빠른 DB는 색인이 많은 DB가 아니라, **실제 질문에 맞는 최소한의 색인** 을 가진 DB입니다.

</details>

### 작동 원리: 계좌 이체가 '반쪽만' 되면 어떻게 될까 - ACID

은행에서 A가 B에게 만 원을 보냅니다. 컴퓨터가 할 일은 둘입니다 - **① A에서 만 원 빼기, ② B에 만 원 넣기.** 그런데 ①을 하고 ②를 하기 직전에 정전이 나면? A 돈은 사라졌는데 B는 못 받았습니다. 만 원이 **증발** 한 거죠.

데이터베이스는 이걸 막으려고 여러 동작을 **하나의 묶음(트랜잭션)** 으로 처리합니다. 원칙은 딱 하나 - **"전부 되거나, 전부 안 되거나."** 중간에 반쪽만 남는 일은 없습니다. 이 성질을 포함한 네 가지 약속을 묶어 **ACID** 라고 부릅니다(1983년에 이름 붙음).

- **A(원자성)** - 반쪽 실행은 없다. 전부 아니면 전부 취소.
- **C(일관성)** - 규칙에 어긋난 상태로는 안 끝난다.
- **I(고립성)** - 동시에 여러 명이 만져도 서로 꼬이지 않는다.
- **D(지속성)** - 한번 됐다고 하면, 정전이 나도 남아 있다.

> 여러분이 외울 건 없습니다. 다만 **돈·주문·예약처럼 "틀리면 큰일"인 데이터** 는 이 성질이 강한 SQL DB가 안전하다 - 이 감만 있으면 충분합니다. (출처: Wikipedia, "ACID"; Härder & Reuter, 1983)

### 조금 더: 동시에 눌러도 안 꼬이는 법 - 트랜잭션

친구 둘이 마지막 티켓 한 장을 거의 동시에 누르면? 아무 규칙이 없으면 둘 다에게 팔릴 수 있습니다. **트랜잭션**(여러 작업을 '전부 되거나 전부 안 되거나'로 묶기)이 이걸 막습니다. 여러분이 기억할 실전은 하나 - **돈·재고·예약처럼 '동시에 두 번' 눌릴 수 있는 곳** 은 AI에게 *"동시에 두 명이 마지막 하나를 요청하는 상황도 테스트하고 안전하게 처리해 줘"*라고 요구하세요.

> 그리고 **ACID는 백업이 아닙니다** - 실수로 지운 걸 되돌려 주진 않으니, 백업은 따로 챙겨야 합니다.

<details class="lms-extra">

<summary><b>더 깊이</b> (개발자로 더 갈 사람만)</summary>

ACID = 원자성(반쪽 실행 없음)·일관성(규칙 유지)·고립성(동시 작업이 서로 안 꼬임)·지속성(커밋되면 장애에도 남음). '고립' 수준에 따라 더러운 읽기·반복 불가 읽기·유령 읽기 같은 현상이 생길 수 있고, 실제 동작은 DB마다 다릅니다(PostgreSQL은 표준 표와 차이가 있어, 이름을 외우기보다 공식 문서와 동시성 테스트로 확인). 재고 차감은 '읽고 → 앱에서 계산 → 쓰기'보다 `UPDATE ... SET stock=stock-1 WHERE id=? AND stock>0`처럼 **조건을 DB에서 함께 확인** 하고 영향받은 행 수를 검사하는 게 경쟁에 강합니다(앱 메모리에서만 검사하면 lost update 위험). 초보 프로젝트의 최소 기준: 데이터 변경을 **짧은 트랜잭션 하나** 로 묶고, DB **제약조건** 을 선언하고, 실패를 **정직하게 안내** 하고, **백업 복구** 를 문서로 남기기.

</details>

### 설계 원칙: 같은 걸 두 번 적지 않기 - '정규화'라는 정리정돈

단골 손님의 주소가 바뀌었습니다. 만약 그 손님의 주문 100건에 주소를 **매번 따로** 적어 뒀다면? 100군데를 다 고쳐야 하고, 하나라도 놓치면 데이터가 서로 안 맞게 됩니다.

그래서 관계형 DB는 **"같은 정보는 한 곳에만 적고, 나머지는 그걸 가리키게"** 합니다. 손님 정보는 '회원 표'에 한 번만 두고, 주문 표에는 '몇 번 회원'이라고만 적는 식이죠. 주소가 바뀌면 회원 표 **한 곳만** 고치면 끝입니다. 이렇게 중복을 걷어내는 정리정돈을 **정규화(normalization)** 라고 부릅니다 - 앞서 코드가 꿈꿨던 "중복 없이"가 바로 이겁니다.

> 이것도 AI가 알아서 해 줍니다. 여러분은 과제에서 데이터의 '모양'을 그릴 때, **"이 정보, 여기저기 똑같이 적히고 있진 않나?"** 한 번만 의심해 보면 됩니다. (출처: Wikipedia, "Database normalization")

### 데이터가 커지면 생기는 일 - 색인과 '마이그레이션'

처음엔 잘 돌던 서비스도 데이터가 쌓이면 조회가 눈에 띄게 느려집니다. 원하는 줄을 찾느라 수백만 줄을 처음부터 훑기 때문인데, 이때의 해법이 앞서 본 **색인(인덱스)** 입니다 - "자주 찾는 칸에 책 색인을 붙인다."

또 하나, 서비스가 자라면 **표 구조 자체를 바꿔야 할 때** 가 옵니다(칸 추가, 표 분리 등). 이미 데이터가 들어 있는 DB의 구조를 안전하게 바꾸는 작업을 **마이그레이션(migration)** 이라 부릅니다. "DB 구조 변경을 Git처럼 이력으로 관리하는 것"이라 생각하면 됩니다 - 되돌리기도 됩니다. (출처: Prisma, "What are database migrations")

> 딱 하나만 기억하세요 - 칸을 **삭제** 하는 변경은 그 안의 데이터가 영구히 사라질 수 있습니다. 그래서 **구조를 바꾸기 전엔 반드시 백업.** 여러분은 "느려지면 색인, 구조를 바꾸면 마이그레이션이라는 게 있다"는 감만 있으면 충분합니다.

### 개념 정리: NoSQL도 한 종류가 아니다 - 네 가지 갈래

"NoSQL"이라고 뭉뚱그렸지만, 사실 성격이 다른 네 종류가 있습니다. 이름과 '어떤 서비스가 쓰나'만 알아 두면 충분합니다. (출처: MongoDB, "NoSQL Databases Explained"; AltexSoft)

| 종류 | 한 줄 설명 | 대표 | 어떤 곳에 |
|---|---|---|---|
| **문서(Document)** | JSON처럼 자유로운 덩어리로 저장 | MongoDB | 형태가 자주 바뀌는 데이터, 프로필·카탈로그 |
| **키-값(Key-Value)** | "키 → 값" 아주 단순·초고속 | Redis, DynamoDB | 장바구니·세션·캐시(빠른 임시 저장) |
| **컬럼(Column)** | 대량 분석에 강한 열 중심 | Cassandra, HBase | 로그·대규모 통계 분석 |
| **그래프(Graph)** | 점과 선으로 '관계'를 저장 | Neo4j | **소셜 네트워크**("친구의 친구") |

재미있는 예 - **그래프 DB** 는 "친구의 친구의 친구"처럼 연결을 따라가는 데 최적이라 소셜 네트워크가 즐겨 씁니다. 반면 **키-값** 은 너무 단순해서 오히려 엄청나게 빠르고, 그래서 쇼핑몰 장바구니 같은 데 쓰입니다.

> 여러분이 지금 이걸 다 고를 필요는 없습니다. 첫 서비스는 SQL로 시작하고, "내 데이터가 소셜 관계 중심이네" "엄청 빠른 임시 저장이 필요하네" 같은 **특별한 이유가 생겼을 때** NoSQL을 후보에 올리면 됩니다.

### 그래서 나는 뭘 쓰나 - 한 줄 기준

- 데이터가 **표로 잘 정리되고 관계가 분명** 하면(사용자·주문·상품처럼) → **SQL(관계형)**
- 형태가 **자주 바뀌고 아주 대규모** 면 → **NoSQL**

**첫 서비스 기본값은 SQL, 그중에서도 PostgreSQL** 이 무난합니다. 자료가 가장 많고, 필요하면 유연한 형식도 지원해서 나중에 곤란해질 일이 적습니다. (출처: TheLinuxCode, "SQL vs NoSQL in 2026"; DEV, "When to Use SQL vs NoSQL 2026") - 물론 정답은 내 데이터의 모양에 달려 있습니다.

### 미니 퀴즈 - 이 데이터, 어떻게 다룰까?

지금까지 배운 걸로 골라 보세요.

| 상황 | SQL? NoSQL? 무엇이 어울릴까 |
|---|---|
| 사용자·주문·상품이 서로 관계로 얽힌 쇼핑몰 |  |
| "친구의 친구"를 따라가는 소셜 관계 |  |
| 잠깐 쓰고 버리는 장바구니(초고속·단순) |  |

<details class="lms-answer">

<summary>정답 보기</summary>

- 관계로 얽힌 쇼핑몰 → **SQL(관계형)**
- 소셜 관계 → **그래프 NoSQL**(예: Neo4j)
- 초고속 장바구니 → **키-값 NoSQL**(예: Redis)

오늘 배운 것들이 그대로 정답입니다.

</details>

## 회원가입과 로그인 - 사용자를 다루기

## 회원가입과 로그인 - 서비스에 '사용자'가 생기면

서비스에 사람이 로그인하기 시작하면 다룰 것이 훅 늘어납니다 - 회원을 **어떻게 저장하고**, 로그인 상태를 **어떻게 유지하며**, 그걸 **빠르고 안전하게** 하느냐. 데이터베이스 강의에서 이걸 짚는 건, 결국 전부 **데이터를 다루는 일** 이기 때문입니다.

인증은 사용자가 누구인지 확인하고, 인가는 그 사용자에게 허용할 일을 정하는 것입니다. 여기서는 **데이터·저장 관점** 으로 이어 봅니다 - 회원 표, 로그인 상태 유지(세션 vs 토큰), 그리고 그걸 빠르게 하는 캐시까지.

### 회원가입·로그인을 '데이터'로 보면

로그인·회원가입은 화면 기능 같지만, 뿌리는 **데이터** 입니다. 먼저 그 데이터가 **DB에 어떻게 담기는지** 부터 봅시다.

먼저 **회원 표** 하나가 필요합니다 - 대충 이런 모양이죠.

| 회원(users) 표 | 예시 한 줄 |
|---|---|
| id · email · password_hash · created_at(가입일) | 1 · hong@example.com · (해시값) · 2026-03-02 |

- **회원가입 = 회원 표에 행 하나 넣기(Create).** 사용자가 입력한 이메일로 새 줄을 만듭니다.
- **로그인 = 그 이메일로 행을 찾아(Read) 비밀번호를 대조.** 맞으면 통과, 틀리면 거절. 결국 앞서 본 **CRUD** 의 조합입니다.

여기서 꼭 지킬 세 가지 - 회원 데이터를 안전하게 저장하기 위한 원칙입니다.

1. **비밀번호는 절대 원문으로 저장하지 않는다.** 칸에 원문이 아니라 **해시(비밀번호를 단방향으로 변환한 값)** 를 저장합니다. 솔트는 비밀번호마다 더하는 무작위 값이고, bcrypt는 비밀번호 저장에 쓰는 해시 함수입니다. 칸 이름을 `password`가 아니라 **`password_hash`** 로 두면 "원문 금지"를 스스로 상기하기 좋습니다.
2. **이메일은 '유일'하게.** 같은 이메일로 두 번 가입되지 않도록 email 칸에 **유일(unique) 제약** 을 겁니다. 없으면 중복 계정·로그인 혼란이 생깁니다.
3. **필요한 것만 모은다.** 가입에 꼭 필요한 것만 받는 것(최소 수집)이 중요해요.

> 주의 - **인증은 직접 만들지 마세요.** 로그인·비밀번호 재설정·소셜 로그인은 함정이 많아, 자작하면 사고로 이어지기 십상입니다. 검증된 **Auth 서비스/라이브러리**(예: Supabase Auth)를 쓰는 게 안전합니다. AI에게도 이렇게 - *"인증은 처음부터 만들지 말고 검증된 Auth를 붙여줘. 비밀번호는 해시로, 이메일은 유일하게."* (출처: OWASP Authentication Cheat Sheet)

**로그인 뒤에는 사용자별 접근도 제한해요.** 로그인했다는 이유만으로 모든 데이터를 읽거나 바꾸게 하면 안 돼요. 서버와 DB에서 요청한 데이터가 그 사용자의 것인지 확인하도록 인증 도구와 접근 규칙을 함께 설정해요.

### 로그인 상태는 어떻게 유지되나 - 세션과 토큰(JWT)

로그인 상태의 발급·검증·만료 처리는 검증된 인증 도구에 맡겨요. 세부 방식이 궁금하면 아래 내용을 펼쳐 봐요.

<details class="lms-extra">

<summary>선택 읽기: 세션과 토큰의 차이</summary>

회원 데이터를 만들었으면 다음 질문은 이겁니다 - **한 번 로그인하면 페이지를 넘겨도 왜 계속 로그인 상태일까?** HTTP(브라우저와 서버가 요청·응답을 주고받는 통신 규칙)는 이전 요청을 자동으로 기억하지 않는 '무상태'라 매 요청이 남남이어서, 매번 '나 아까 그 사람'을 다시 증명해야 합니다.

그 증명을 실어 나르는 게 **쿠키** 입니다 - 사이트가 브라우저에 건네는 작은 메모로, 브라우저가 보관했다가 **요청할 때마다 자동으로 함께 보냅니다**. 로그인이 계속 유지되는 건 이 쿠키 덕분이죠. 그럼 이 쿠키(또는 그 자리)에 **무엇을 담느냐** - 여기서 두 갈래가 갈립니다.

**갈래 1 - 세션 (서버가 기억한다)**
서버가 로그인 상태를 자기 쪽 '세션 저장소'에 적어 두고, 브라우저엔 그걸 가리키는 **번호표(세션 ID)** 만 쿠키로 줍니다. 서버가 상태를 쥐고 있어서 **로그아웃·강제 차단을 바로 할 수 있습니다** - 저장소에서 지우면 번호표는 즉시 휴지 조각이 되니까요. 단점은 서버가 여러 대로 늘면 그 저장소를 **함께 나눠 써야** 한다는 것입니다. (출처: MDN, "Session management")

**갈래 2 - 토큰 (JWT, 서버가 기억하지 않는다)**
로그인하면 서버가 사용자 정보와 만료 시각을 담아 **위조 못 하게 서명(도장)한 증명서(JWT)** 를 통째로 내줍니다. 이후 브라우저는 요청마다 이 토큰을 (쿠키나 요청 헤더에 담아) 내밀고, 서버는 서명과 만료 시각 등 필요한 조건을 검증해요(이 검증은 인증 도구에 맡겨요). 서버 확장·여러 서비스 연동에 강합니다.

- **꼭 알 것:** 토큰 내용은 **암호가 아니라 그냥 인코딩** 이라, 가진 사람은 누구나 열어 봅니다. 서명이 막는 건 '못 바꾸게'이지 '못 보게'가 아니에요 - 그래서 **민감정보(비밀번호 등)를 토큰에 넣으면 안 됩니다.**
- **단점:** 한 번 내준 토큰은 **즉시 무효화가 어렵습니다**(서버에 지울 기록이 없으니). 그래서 보통 **만료를 짧게 두고 '리프레시 토큰'으로 조용히 갱신** 하는 방식을 씁니다. (출처: Auth0, "JSON Web Tokens")

**그래서 뭘 쓰나** - 단순한 웹앱이고 즉시 로그아웃·차단이 중요하면 **세션** 이 더 단순·안전하고, 여러 서비스·모바일·큰 확장이 필요하면 **토큰** 이 유리합니다.

> 가장 중요한 실용 결론 - **이걸 직접 밑바닥부터 만들지 마세요.** MDN조차 "세션 관리를 직접 구현하지 말고 검증된 라이브러리를 쓰라"고 못 박습니다. Auth 서비스(Supabase Auth·Clerk·Auth0 등)가 세션이든 토큰이든 발급·만료·갱신·저장까지 알아서 안전하게 처리합니다. 우리 몫은 **"무슨 방식인지 이해하고, 믿을 만한 도구에 맡기는"** 판단까지입니다. (출처: MDN, "Session management")

</details>

### 캐시와 Redis - 로그인이 빠른 비결, 그리고 저장 위치

비밀번호 원문을 저장하지 않고, 검증된 인증 도구를 사용하며, 사용자별 접근을 제한하는 것이 본문의 핵심이에요. 저장 위치와 성능 이야기는 선택해서 읽어요.

<details class="lms-extra">

<summary>선택 읽기: 캐시·Redis와 로그인 증표 저장 위치</summary>

**캐시(cache)** 는 자주 쓰는 데이터를 **메모리에 잠깐 두어 빠르게** 꺼내 쓰는 것입니다. 매번 창고(디스크·DB) 깊숙이 다녀오는 대신, 자주 찾는 물건을 계산대 옆 바구니에 올려 두는 셈이죠. 속도가 크게 오르는 대신 **원본과 잠깐 어긋날 수 있다** 는 트레이드오프가 있어서, '언제까지 신선한 걸로 볼지(수명, TTL)'를 정해 둡니다. (출처: MDN, "HTTP caching")

로그인에서 캐시가 등장하는 자리가 바로 위의 **세션 저장소** 입니다. 세션 확인은 거의 모든 요청마다 일어나 '빠름'이 생명인데, 앞서 NoSQL에서 본 **Redis(키-값, 메모리 기반)** 가 딱 맞습니다 - 조회가 매우 빠르고, **키마다 자동 만료(TTL)** 를 걸 수 있어 '30분 활동 없으면 자동 로그아웃' 같은 걸 손쉽게 만듭니다. (출처: Redis 공식, "Session management") 그래서 "세션을 여러 서버가 나눠 쓰는 문제"와 "캐시로 속도를 얻는 것"이 **같은 도구(Redis) 하나** 로 만나죠.

> 보안 메모 - 브라우저에서 로그인 증표를 **어디에 두느냐** 도 중요합니다. `localStorage`는 악성 스크립트(XSS)에 그대로 읽혀 위험해서, 보안 표준(OWASP)은 **자바스크립트가 못 읽는 `httpOnly` 쿠키** 를 권합니다. 여러분은 세 줄만 기억하세요 - **① 저장·갱신은 Auth 서비스에 맡기고, ② 민감정보는 토큰에 넣지 말고, ③ 저장 위치는 안전한 곳으로.** (출처: OWASP HTML5 Security Cheat Sheet)

</details>

## 데이터를 안전하게 - 보안과 백업

## 데이터는 모아 두는 순간 '지켜야 할 것'이 된다

---

데이터베이스에 진짜 정보가 쌓이기 시작하면, 두 가지 위험이 따라옵니다 - **누가 훔쳐 가는 것(보안)** 과 **실수로 날아가는 것(백업)**. 코드는 AI가 쓰지만, 이 위험을 아는 건 여러분의 몫입니다.

회원 가입 화면에 이메일과 비밀번호를 입력한 뒤, 개발자는 그 정보가 어디로 가고 누가 볼 수 있는지 점검하지 않았습니다. 며칠 뒤 낯선 사람이 다른 회원의 주문 내역을 조회했다는 문의가 들어왔습니다. 데이터를 모으는 순간부터 입력값을 안전하게 처리하고, 허용된 사용자만 데이터에 접근하도록 제한해야 합니다.

![파라미터화 쿼리: SQL 문장과 입력값을 별도 통로로 전달하는 그림](https://ebhtmardjzsiyxjncvbb.supabase.co/storage/v1/object/public/images/01-vibe-coding-web/day08_infographic_paramquery.png)
*안전한 쿼리는 SQL 문장과 입력값을 별도 통로로 전달한다*

### 보안 이야기: "Little Bobby Tables" - SQL 주입

프로그래머 사이에서 전설적인 만화가 있습니다(xkcd, 2007). 학교에서 엄마에게 전화가 옵니다.

> **학교**: 아드님 이름을 데이터베이스에 넣었더니 학생 기록이 **전부 지워졌어요.**
> **엄마**: 아, 우리 아들 이름이 좀 특이하죠. `Robert'); DROP TABLE Students;--` 라서요.
> **엄마**: 앞으로는 **입력값을 꼭 소독(검증)** 하시길 바라요.

무슨 일이 벌어진 걸까요? 아들 이름에 **DB 명령어처럼 생긴 문자** 가 들어 있었고, 학교 프로그램이 그걸 **이름이 아니라 명령으로 실행** 해서 `학생(Students)` 표를 통째로 **삭제(DROP)** 해 버린 겁니다. 이것이 그 유명한 **SQL 주입(SQL Injection)** - 25년 넘게 보안 위협 1위권을 지키는 공격입니다. 로그인 창에 특정 문자를 넣어 **비밀번호 없이 로그인** 하는 것도 같은 원리입니다. (출처: xkcd "Exploits of a Mom"; Malwarebytes, "What is SQL injection")

> 만화 원본: https://xkcd.com/327/ · 언어별로 **막는 법** 을 정리한 곳: https://bobby-tables.com/

### 방어법 - "레시피와 재료를 섞지 않는다"

SQL 주입은 무서워 보여도 **정해진 방어법이 확실합니다**. 핵심은 하나 - **명령(레시피)과 사용자 입력(재료)을 섞지 않는 것** 입니다.

- 위험: 사용자가 친 글자를 명령 문장에 **그대로 이어 붙이기** → DB가 "어디까지가 명령이고 어디부터가 데이터"인지 헷갈림
- 안전: **자리표(`?`)를 미리 비워 둔 명령** 을 만들고, 사용자 입력은 **따로** 건네기 → DB가 입력을 **무조건 '그냥 데이터'로만** 취급. `Robert'); DROP...`을 넣어도 그런 **이름을 가진 사람을 찾을 뿐** 명령으로 실행하지 않음

이 방식을 **파라미터화 쿼리(prepared statement)** 라고 합니다. 보안 전문가들이 꼽는 표준 방어법입니다. (출처: OWASP, "SQL Injection Prevention Cheat Sheet")

> AI에게 이렇게 요구하세요 - **"DB에 넣는 값은 문자열로 이어 붙이지 말고, 파라미터화 쿼리(prepared statement)로 처리해줘. 사용자 입력을 그대로 신뢰하지 마."** 한 문장이 학생 기록을 지키는 셈입니다.

### 실화: 알려진 구멍을 안 막아 십대에게 뚫린 통신사

SQL 주입이 옛날이야기 같나요? 2015년, 영국의 대형 통신사 **TalkTalk** 이 이 공격으로 뚫렸습니다. 놀라운 건 공격자가 대단한 조직이 아니라 **십대들** 이었다는 점입니다. 심지어 이용된 구멍은 **3년 반 전에 이미 고치는 패치가 나와 있던** 것이었죠 - 회사가 그걸 안 막고 방치한 겁니다.

결과는 뼈아팠습니다. 고객 **약 15만 7천 명** 의 정보가 새어 나갔고(그중 1만 5천여 명은 계좌 정보까지), 체포된 이들은 모두 **21세 미만** 이었습니다. 회사는 규제기관에서 당시 최고액 벌금을 맞았고, BBC와 TalkTalk 결산 기준 사고 비용은 **약 4,200만 파운드** 였습니다. 규제기관의 총평은 짧고 매서웠습니다 - *"가장 기본적인 보안 조치조차 하지 않았다."* (출처: 영국 정보위원회(ICO) 공식 발표; BBC, "TalkTalk hack costs reach £42m", 2016. https://www.bbc.co.uk/news/business-38262235)

규모로는 더 큰 사고도 있습니다. 2008년 미국 결제회사 **Heartland** 는 같은 SQL 주입으로 카드번호 **약 1억 3천만 장** 이 털렸습니다 - 당시 사상 최대 규모였죠. (출처: Wikipedia, "Albert Gonzalez")

> 교훈은 앞의 'Bobby Tables'와 똑같습니다. 대부분의 큰 사고는 새로운 마법이 아니라, **오래전에 답이 나온 기본을 안 지켜서** 벌어집니다. 다행히 요즘 AI와 최신 도구는 이 방어를 기본으로 해 주지만, **"내 서비스도 이게 되어 있나?"** 묻는 습관은 사람의 몫입니다.

### 개인정보 다루기 - 모을수록 무거워진다

데이터베이스에 이름·이메일·비밀번호 같은 **개인정보(PII)** 가 쌓이면, 그건 자산이자 **책임** 이에요. 이메일과 비밀번호가 유출되면 다른 서비스의 계정까지 위험해질 수 있어요.

야후 보이스(2012)는 비밀번호를 그대로(평문) 저장했다가 유출 사고를 겪었어요. 평문 비밀번호가 유출되면 공격자가 그 값을 바로 읽고 사용할 수 있어요. 같은 비밀번호를 다른 서비스에서도 썼다면 피해가 번질 수 있어요.

이런 위험을 줄이기 위해 어떤 정보를 모으고 어떻게 다룰지, 다음의 기본 원칙으로 정리해 봐요.

### 개인정보의 최소 원칙 - 덜 모으고, 조심해서 다룬다

사용자 정보를 저장하는 순간, 처음 시작해도 지켜야 할 기본 네 가지가 있습니다.

1. **필요한 것만 모은다(최소 수집).** 한국 개인정보보호법도 "목적에 필요한 최소한만" 모으도록 정합니다. "이메일 하나면 되는데 전화번호·생년월일까지 받고 있진 않나?"를 늘 자문하세요. (출처: 국가법령정보센터, 개인정보 보호법 제3·16조)
2. **비밀번호는 절대 그대로 저장하지 않는다.** 앞서 배운 대로 **해시** 로 바꿔 저장하고, 그것도 빠른 해시가 아니라 bcrypt·Argon2 같은 **비밀번호 전용** 을 씁니다. (출처: OWASP Password Storage Cheat Sheet)
3. **민감정보(주민번호·카드번호)는 직접 다루지 않는 게 최선.** 결제는 결제대행사(PG)에, 본인확인은 인증 서비스에 맡기고 **내 DB엔 저장하지 않습니다.**
4. **모을 때 밝힌 목적에만 쓴다(목적 제한).** 다른 데 쓰거나 남에게 넘기려면 별도 동의가 필요합니다.

> 한 줄 요약 - **"덜 모으고, 비밀번호는 해시하고, 민감정보는 남에게 맡기고, 목적 밖으로 쓰지 않는다."** 모으지 않은 정보는 유출될 수도 없습니다 - 가장 안전한 데이터는 **애초에 안 가진 데이터** 입니다.

**실제 사용자를 받는다면 (로그인·회원가입 기준)** - 최소한 이건 지키세요.

- **가입 때 '수집·이용 동의'를 받는다** - 무엇을·왜 모으는지 알리고 동의(필수/선택 구분).
- **개인정보처리방침 한 장을 둔다** - 무엇을·왜 모으고 얼마나 보관하는지(AI에게 초안을 요청해도 됩니다).
- **제3자에게 넘길 때**(제공)는 미리 알리고 동의를 받되, 반드시 다섯 가지를 밝힙니다 - ① 제공받는 자 ② 이용 목적 ③ 제공하는 항목 ④ 보유·이용 기간 ⑤ 동의 거부권과 거부 시 불이익. (되도록 안 넘기는 게 최선입니다.)

> 근거·자세한 내용은 [개인정보 보호법(국가법령정보센터)](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B0%9C%EC%9D%B8%EC%A0%95%EB%B3%B4%EB%B3%B4%ED%98%B8%EB%B2%95)에서 볼 수 있습니다.

### 데이터를 잃지 않기 - 백업, 그리고 유명한 참사들

해커가 없어도 데이터는 사라집니다 - **실수 한 번** 으로요. 그래서 **백업**(사본을 따로 보관)이 필요합니다. 전설적인 사례로 배워 봅시다.

- **픽사 「토이 스토리 2」(1999년 개봉, 사고는 1998년 제작 중)** - 누군가 잘못된 명령을 실행해 영화 파일이 **눈앞에서 지워지기 시작** 했습니다. 설상가상 백업 시스템도 한 달째 고장 나 있었죠. 영화를 살린 건, 출산 후 재택근무하던 한 기술 총괄(TD)이 **집 컴퓨터에 두었던 2주 전 사본** 이었습니다. 하마터면 영화가 통째로 날아갈 뻔했습니다.
- **GitLab(2017)** - 엔지니어가 실수로 잘못된 서버의 데이터를 지웠고, **여러 백업이 모두 조용히 고장 나 있어** 6시간치 데이터를 잃었습니다.
- **Ma.gnolia(2009)** - 백업은 있었지만 **원본과 같은 시스템에** 뒀다가, 그 시스템이 망가지자 백업도 함께 사라져 서비스가 문을 닫았습니다.

교훈은 분명합니다.

- **백업이 '진짜 되는지' 가끔 확인한다** (픽사·GitLab 둘 다 '있다고 믿었지만' 고장 나 있었음)
- **사본은 다른 곳에** 둔다 (원본과 같은 곳이면 함께 사라짐)
- **여러 벌** 을 둔다

> 참고 - 요즘 클라우드 DB는 백업을 도와주지만, **무료 요금제에는 자동 백업이 없는 경우가 많습니다**(예: Supabase 무료 요금제는 직접 백업이 필요하고, 자동 일일 백업은 유료 Pro부터 제공됩니다). 그러니 **"내 요금제에 자동 백업이 포함되는지"를 확인** 하고, 없으면 정기적으로 직접 사본을 내려받아 두세요. (출처: Supabase Docs, "Database Backups"; EnvisionIT, "Six Horror Stories of Data Loss")

![자기테이프 라이브러리(대용량 백업 장치)](https://commons.wikimedia.org/wiki/Special:FilePath/StorageTek_Powderhorn_tape_library.jpg?width=640)
*대용량 데이터를 오래 보관(아카이빙)하는 데는 지금도 자기테이프(LTO)가 많이 쓰입니다 - 값싸고 수십 년 가며, 인터넷과 떨어져 있어 랜섬웨어에 강하거든요(CERN도 수백 PB를 테이프에 보관). 단, 이건 데이터센터 규모 이야기이고, 개인·작은 서비스 백업은 클라우드·외장하드면 충분합니다. (이미지: Wikimedia Commons)*

### 백업의 황금률 - 3-2-1

백업은 "했다"가 아니라 **"복구된다"** 가 중요합니다. 미국 사이버보안기관(CISA)이 권하는 표준이 **3-2-1 규칙** 입니다.

- **3벌** - 원본 1 + 사본 2, 최소 세 벌을 둔다.
- **2종** - 서로 다른 매체 두 곳에(예: 노트북 디스크 + 클라우드). 한 종류가 통째로 망가져도 다른 쪽은 살아남게.
- **1벌은 다른 장소에(off-site)** - 화재·도난·랜섬웨어처럼 **한 장소를 통째로 잃는 사고** 에 대비.

개인이 실천하는 예: ①노트북의 원본 ②외장하드 사본 ③클라우드(구글 드라이브나 DB 자동 백업)에 한 벌 - 이러면 자연스럽게 3벌·2종·off-site 1벌이 갖춰집니다. (출처: CISA, "Data Backup Options")

> 그리고 CISA가 특히 강조하는 것 - **복구가 진짜 되는지 가끔 확인하세요.** 앞서 픽사·GitLab 사고가 보여줬듯, **열어본 적 없는 백업은 백업이 아닙니다.**

### 미니 퀴즈 - 이 습관은 안전할까요?

보안과 백업에서 배운 내용을 떠올리며 O/X를 골라 봐요.

| 습관 | 안전할까? (O/X) |
|---|---|
| 사용자가 친 값을 명령문에 그대로 이어 붙인다 |  |
| 비밀번호를 그대로(평문) 저장한다 |  |
| 정기적으로 백업(자동 또는 직접)을 해 둔다 |  |

<details class="lms-answer">

<summary>정답 보기</summary>

- 입력을 그대로 이어 붙이기 → **X.** SQL 주입 위험이 있어요. 파라미터화로 입력값과 명령문을 분리해요.
- 비밀번호 평문 저장 → **X.** 야후 보이스의 교훈이에요. 비밀번호 전용 해시를 쓰는 검증된 인증 도구에 맡겨요.
- 정기 백업 → **O.** 토이 스토리 2의 교훈이에요. 데이터를 잃을 때 복구할 사본이 필요하며, 실제로 복구되는지도 확인해요.

</details>

## 무료로 쓸 수 있는 DB 고르기

## 공짜로 시작하기 - 무료 데이터베이스 고르기

---

프로젝트를 막 시작한 학생이 "무료로 시작하고 싶은데 뭘 골라야 하지?"라며 서비스마다 다른 제한을 비교하고 있습니다. 가입은 쉬워도 저장 용량이나 사용량 제한을 놓치면, 발표 직전에 앱이 멈출 수 있습니다. 그래서 목적과 제한을 함께 보고 고르는 기준이 필요합니다.

처음 시작해도 **무료로 진짜 데이터베이스** 를 쓸 수 있습니다. 다만 시작하기 전에 한 가지만 기억하세요 - **무료 티어는 배우기·MVP(초기 버전)용이지, 큰 규모의 운영용은 아닙니다.** (출처: agentdeals, "Database Free Tier Comparison 2026")

### 무료 데이터베이스 비교 (2026-02 공식 가격 페이지 확인, 정책은 바뀔 수 있음)

“무료로 시작하고 싶은데 뭘 골라야 하지?” 기능을 만들기 전부터 표가 부담스러울 수 있습니다. 먼저 **내 서비스가 SQL 표와 로그인·파일저장을 한 번에 원하는지**, 아니면 DB만 필요한지를 적어 보세요. 무료 한도는 가입 직전에 공식 가격 페이지에서 다시 확인합니다.

| 서비스 | 종류 | 공식 무료/시작 정책에서 확인한 점 | 처음 시작할 때 |
|---|---|---|---|
| **Supabase** | PostgreSQL (SQL) | Free 프로젝트당 DB 500MB, 스토리지 1GB. 자동 백업은 Free에 포함되지 않으며 1주 미사용 뒤 일시정지될 수 있음 | 로그인·파일까지 한 번에 실습할 때 후보 |
| **Neon** | PostgreSQL (SQL) | Free는 개발·프로토타이핑용이며, 유휴 컴퓨트가 scale-to-zero될 수 있음 | 순수 Postgres와 브랜치를 써 보고 싶을 때 |
| **Turso** | SQLite | Free: DB 100개, 저장소 5GB, 월 읽기 5억 행·쓰기 1천만 행, PITR 1일 | SQLite 기반의 가벼운 서비스를 실험할 때 |
| **PlanetScale** | Postgres / Vitess | 현재 시작 요금은 개발용 Postgres 단일 노드 월 $5부터로, 무료 티어 표에는 넣지 않음 | 무료만이 조건이면 다른 후보부터 검토 |

> **확인 링크(2026-02)**: Supabase https://supabase.com/pricing · Neon https://neon.com/pricing · Turso https://turso.tech/pricing · PlanetScale https://planetscale.com/pricing. 정책·지역·가입 시점에 따라 한도와 기능이 달라질 수 있습니다.

### 클라우드 말고, 내 컴퓨터에 직접 설치해서 무료로

지금까지는 '남의 컴퓨터(클라우드)'를 빌리는 무료 티어를 봤죠. 그런데 DB는 **내 컴퓨터에 직접 설치** 해서도 무료로 쓸 수 있습니다. 대부분 오픈소스라 설치도 사용도 공짜예요.

- **SQLite - 설치도 서버도 없다.** DB 전체가 **파일 하나** 입니다. 서버를 따로 켤 필요 없이 그 파일을 앱이 직접 읽고 씁니다. '가장 가볍게 시작'하기 좋아 작은 앱·프로토타입·모바일 앱이 즐겨 씁니다. (파이썬 등 많은 도구에 기본 내장돼 있어 사실상 '설치 없이' 쓰기도 합니다.)
- **PostgreSQL · MySQL - 진짜 DB를 내 컴퓨터에.** 회사들이 실제로 쓰는 그 오픈소스 DB를 내 컴퓨터에 무료로 설치할 수 있습니다. 클라우드에서 빌리던 것과 같은 것을 내 손안에서 돌리는 거죠.
- **Docker - 명령 한 줄로 띄우기.** 설치가 번거로우면 Docker로 DB를 '컨테이너'로 띄웁니다. 내 컴퓨터를 어지럽히지 않고 명령 한 줄로 켜고, 필요 없으면 통째로 지웁니다.

**로컬(내 컴퓨터) vs 클라우드, 언제 뭘?**

- **로컬** - 완전 무료, 오프라인에서도 되고, 내 데이터가 내 손안에. **개발·학습·실험** 에 최고. 단, 남에게 열려면 결국 배포하거나 터널이 필요하고, 내 컴퓨터가 꺼지면 멈춥니다.
- **클라우드** - 24시간 켜져 있고 남에게 안정적으로 열립니다. **실제 서비스** 엔 이쪽.

> 그래서 흔한 흐름은 - **내 컴퓨터(로컬)에서 만들고 → 세상에 낼 때 클라우드로.** AI에게 이렇게 부탁할 수 있습니다 - *"개발용으로는 가벼운 로컬 SQLite로 시작하고, 배포할 때 클라우드 DB로 바꿀 수 있게 해줘."*

### 무료 티어, 시작 전에 확인할 것

- **용량 한도** - 예: 500MB. 이미지 같은 큰 파일을 많이 넣으면 금방 찹니다.
- **미사용 시 일시정지** - 예: Supabase는 1주일 안 쓰면 프로젝트가 잠깐 멈춥니다(다시 켜면 됨).
- **백업 방법** - 앞에서 본 참사를 피하려면, 자동 백업이 되는 요금제인지(무료면 직접 백업이 필요한지) 꼭 확인하세요.
- **정책 변경** - 위 PlanetScale처럼 무료가 사라질 수 있습니다.
- **트래픽이 늘면 유료** - 서비스가 커지면 자연스럽게 유료로 넘어갑니다. **요금·용량 알림** 을 켜 두세요.

### AI에게 데이터베이스를 붙여 달라고 하는 프롬프트

문법을 몰라도, **무엇을 저장할지 + 조건** 만 주면 AI가 스키마 설계부터 연결·보안까지 도와줍니다.

```text
내 서비스에 데이터를 저장하려고 해. 나는 처음 배우고 무료로 시작하고 싶어.
1. 내가 저장할 것은 [예: 상품, 주문]인데, 어떤 표(테이블)와 항목(열)이 필요할지 스키마를 제안해줘.
2. 무료 SQL 데이터베이스(예: Supabase, PostgreSQL) 기준으로 이 스키마를 만드는 방법을 알려줘.
3. 접속 정보(비밀번호·키)는 코드에 넣지 말고 환경변수로 빼줘.
4. DB에 값을 넣을 때는 파라미터화 쿼리로 처리하고, 비밀번호는 해시로 저장해줘.
```

환경변수는 코드 밖에서 프로그램에 전달하는 설정값입니다. 접속 비밀번호 같은 비밀값을 코드에 직접 적지 않도록 분리하고, 오늘 배운 **SQL 주입 방어** 도 함께 요청합니다.

## 실습·과제와 마무리

## 실습: 에이전트로 '내 Todo 앱'을 내 컴퓨터에 만들기 (따라 하기)

이제 배운 걸 모아 **직접 만들어 봅니다.** 만들 것은 **내 컴퓨터에서 도는 작은 Todo(할 일) 앱** - 할 일을 추가·확인·완료·삭제하고, 그 데이터가 **내 컴퓨터의 무료 DB** 에 저장돼 껐다 켜도 남습니다. Todo 앱은 **저장·조회·수정·삭제(CRUD)** 의 교과서라 DB를 손에 익히기에 딱입니다.

방식은 **에이전틱** - 목표(PRD, 만들 기능과 조건을 적은 문서)를 주면 에이전트가 계획·설치·구현을 하고, 우리는 검토·판단·수정합니다.

**길 A에서 로컬 저장을 확인한 뒤, 길 B에서 클라우드로 전환해요. 오늘 실습은 길 B까지 마치면 완성이에요.**
- **길 A - 내 컴퓨터(로컬)**: 무료 로컬 DB(예: SQLite)에 저장. 인터넷 없이 내 PC에서 완결됩니다.
- **길 B - Supabase로 저장소 옮기고 Render 배포 안내받기**: 같은 앱의 저장소를 클라우드 DB로 바꾸고, 백엔드를 클라우드에 올리는 데 필요한 준비물을 에이전트에게 설명받아요.

**도구** - Claude Code나 Cursor의 에이전트 모드처럼 **명령 실행·파일 생성·설치를 스스로 하는** 도구를 씁니다. AI가 짠 코드는 한 번씩 훑어보세요.

필수 실습은 저장소를 클라우드 DB로 바꾸고, 배포 준비물을 에이전트에게 안내받아 노트에 기록하는 것까지예요. 화면과 백엔드는 기본적으로 내 컴퓨터에서 실행해요. 여유가 있고 더 해 보고 싶다면 실제 배포에 도전해 봐도 좋아요.

**시작 준비**

- 상품 목록 프로젝트가 있다면 그 프로젝트와 별도로 새 Todo 프로젝트 폴더를 만들어요.
- 터미널에서 새 폴더로 이동한 뒤 `claude`를 실행해 Claude Code를 열어요. Cursor를 쓴다면 이 폴더를 열고 에이전트 모드를 사용해요.
- 작업 위치가 새 Todo 폴더인지 확인한 뒤 다음 PRD를 작성해요.

### 1부: PRD로 시작하고, DB 구조를 설계받기

앞서 배운 대로 **PRD(만들 것 한 장)** 부터 써요. 아래 템플릿을 프로젝트 폴더의 `PRD.md`에 복사해 작성해요. Todo 앱의 **기본 기능 + 내가 넣고 싶은 기능** 을 적어요.

```text
[무엇] 내 컴퓨터에서 도는 Todo(할 일) 앱.
[기본 기능(CRUD)] 할 일 추가 / 목록 보기 / 완료 표시 / 삭제
[저장] 데이터는 내 컴퓨터의 무료 DB에 저장(껐다 켜도 남기)
[내가 넣고 싶은 기능]   <- 여기에 1～2개 직접 적기
   예: 마감일 표시 / 카테고리(태그) / 오늘 할 일만 보기 / 검색
[규칙] 무료 도구만. 비밀값은 .env로.
       (로그인 같은 걸 넣는다면 직접 만들지 말고 검증된 라이브러리로.)
```

이 PRD를 에이전트에게 주고, **먼저 DB 구조부터 설계** 받으세요.
> 프롬프트: *"위 PRD로 만들 Todo 앱의 **DB 구조(어떤 표·열·관계)** 를 먼저 설계해서 **구조도로 그려** 보여 줘. 내 커스텀 기능(마감일·태그 등)도 반영해서. 로컬 무료 DB(예: SQLite)를 쓸 거야."*

→ **확인:** AI가 표(예: `todos`, 태그를 넣었다면 `tags`와 연결표)와 열(제목·완료여부·마감일…)을 **구조도** 로 보여 줍니다. 앞서 배운 '표 사이 관계'대로 잡혔는지 눈으로 확인하세요.

![마감일·태그를 넣은 Todo 앱의 DB 구조도 예시: todos와 tags가 연결표 todo_tags로 이어진다](https://ebhtmardjzsiyxjncvbb.supabase.co/storage/v1/object/public/images/01-vibe-coding-web/day08_todo_erd.png)
*예시 - 마감일·태그(카테고리)를 넣은 Todo 앱의 구조도. 할 일(todos)과 태그(tags)는 '연결표(todo_tags)'로 이어집니다(다대다). 이런 그림을 AI에게 받아 확인하면 됩니다.*

### 2부: 길 A - 내 컴퓨터에 만들어 돌리기 (로컬)

DB 구조에 합의했으면, 에이전트에게 **로컬 무료 DB로 실제 구현** 을 맡깁니다.
> 작업 지시서: *"이 PRD와 방금 합의한 DB 구조로 Todo 앱을 만들어 줘. 데이터는 로컬 SQLite 파일 하나(`todo.db`)에 저장하고, 브라우저에서 `localhost`로 열리게 해 줘. DB에 접근하는 코드는 백엔드(서버 쪽) 코드에만 두고, 화면 코드는 백엔드 API에만 요청해 줘. `.env`·`todo.db`는 `.gitignore`에 넣어. 먼저 전체 계획을 보여주고, 각 단계 끝에 뭘 확인하면 되는지 알려 줘."*

에이전트가 계획→설치→파일 생성→DB 만들기까지 스스로 합니다. 우리가 할 일은 셋.
- **계획 한 번 읽기**: PRD와 맞는지 확인해요.
- **명령 내용을 확인한 뒤 승인하기**: 작업 대상이 이 프로젝트 폴더 안인지, 무엇을 지우거나 바꾸는지 확인한 뒤 승인해요. 삭제·전역 설치는 다시 한 번 확인해요.
- **잘 되던 순간마다 Git 커밋하기**: 되돌릴 지점을 남겨요.

→ **통과:** 브라우저에서 할 일을 추가하면 목록에 뜨고, **새로고침·앱 재시작 후에도 남는다.** DB 뷰어로 `todo.db`를 열어 `todos` 표에 행이 실제로 쌓였는지 확인(확인 툴은 AI에게 추천받기 - SQLite는 'DB Browser for SQLite'나 VS Code의 SQLite 확장).

실제로 만들면 이런 화면이 뜹니다.

![실제로 만들어 돌린 Todo 앱 화면: 제목 My Todo, 입력창과 추가 버튼, 마감일이 붙은 할 일 목록](https://ebhtmardjzsiyxjncvbb.supabase.co/storage/v1/object/public/images/01-vibe-coding-web/day08_todo_app_screenshot.png)
*예시 - 실제로 만들어 돌린 Todo 앱 화면. 추가한 할 일이 목록에 남고, 데이터는 내 컴퓨터의 `todo.db`에 저장됩니다.*

**자주 막히는 곳**

| 증상 | 확인 / 해결 |
|---|---|
| `npm install`에서 better-sqlite3 빌드 에러(node-gyp) | 네이티브 모듈이라 빌드 도구가 필요. 에러를 그대로 에이전트에 주고 "내 OS(맥/윈도우) 기준 빌드 도구를 설치·확인해 줘"(맥: Xcode Command Line Tools, 윈도우: 빌드 툴·Python). Node는 LTS 권장 |
| 포트가 already in use(EADDRINUSE) | 이전 서버가 아직 떠 있음. 그 창을 끄거나 다른 포트로 |
| 추가했는데 새로고침하면 사라짐 | 화면에만 있고 DB엔 안 들어감. "추가한 할 일을 DB에 저장하고 다시 읽어오게 해 줘" |
| 코드 고쳤는데 반영 안 됨 | 개발 서버를 껐다 켜기(특히 `.env` 수정 후) |
| `todo.db`를 GitHub에 올려버림 | `.gitignore`에 추가. 이미 커밋했다면 에이전트에게 'git 이력에서도 빼 줘'까지 |

> **안 되면 - 에러를 그대로 에이전트에게** 붙여넣어 고치게 하세요(관찰→수정 반복). 같은 걸 계속 못 고치면 마지막으로 되던 지점으로 되돌리고 문제를 잘게 쪼개 다시 시키세요.

### 3부: 길 B - Supabase로 저장소 옮기고 Render 배포 안내받기

Todo 앱의 화면과 백엔드는 내 컴퓨터에서 실행해요. 길 A에서 확인한 앱의 저장소를 **클라우드 DB(Supabase)** 로 바꾸고, 백엔드를 Render 같은 클라우드 서비스에 올리는 데 필요한 준비물을 에이전트에게 안내받아 노트에 기록해요.

**RLS(행 수준 보안)** 는 누가 어떤 행을 읽고 바꿀 수 있는지 제한하는 규칙이에요. 켜기만 하는 것으로 끝내지 않고, 로그인한 사용자가 자기 할 일에만 접근할 수 있도록 규칙을 설정해요.

앞에서 본 "프론트는 못 믿는다"를 떠올려 봐요. 화면 코드는 누구나 열어 볼 수 있어서 DB 주소와 접속 키를 여기에 두면 다른 사람이 DB에 직접 요청할 수 있으니, 오늘 DB를 부르는 경로의 원칙은 **프론트는 백엔드에만 말하고, 백엔드가 DB에 말해요.**

접속 키를 코드에 직접 적어 두면, 코드를 GitHub 등에 올릴 때 키도 함께 새어 나갈 수 있어요.
그래서 키는 `.env` 파일에 두고, 백엔드 코드는 환경변수 이름으로만 읽게 해요.
길 A에서 `.env`를 이미 `.gitignore`에 넣었으니, 키가 든 파일은 Git에 올리지 않도록 해 둔 거예요.

> 프롬프트: *"같은 Todo 앱을, 로컬 SQLite 대신 **Supabase(무료 클라우드 Postgres)** 에 저장하도록 바꿔 줘. 화면(프론트) 코드에서 Supabase를 직접 호출하지 말고, 백엔드 코드에서만 DB에 접근하게 해 줘. 화면은 백엔드 API에만 요청하고, 접속 키도 백엔드 쪽에서만 쓰게 해 줘. 접속 키는 `.env`로 빼고, **RLS(행 수준 보안)를 켜서** 허용된 접근만 되게 해 줘. Supabase Auth로 로그인하고, 할 일마다 소유 사용자를 기록해 줘. 로그인한 사용자가 자기 할 일만 조회·추가·수정·삭제할 수 있도록 RLS 정책도 설정해 줘. 전환이 끝나면 이 백엔드를 Render 같은 클라우드 서비스에 올리는 데 필요한 준비물(코드 보관 위치, 환경변수, 시작 명령 등)을 알려 줘. 내가 배포에 도전하겠다고 하면 단계를 안내하며 실제 배포까지 함께 진행해 줘."*

**확인**

- AI에게 DB에 접근하는 코드가 어떤 파일에 있는지 짚어 달라고 하고, 그 코드가 브라우저가 아니라 서버에서 실행되는 이유도 설명받아요. 그 설명이 앞의 원칙과 맞는지 확인해요.
- Supabase 대시보드의 **Table Editor** 에서 `todos` 표에 행이 쌓이는지 확인해요.
- 앱에서 할 일을 추가하고 새로고침한 뒤에도 남는지 확인해요.
- 서로 다른 계정으로 로그인해 **다른 계정의 할 일은 보이지 않는지** 확인해요. 다른 계정의 할 일을 수정·삭제하는 요청도 거절되는지 확인해요. AI에게 테스트 계정 두 개를 만드는 방법과, 다른 계정의 데이터를 조회·수정·삭제해 보는 절차를 요청해요. A 계정으로 만든 할 일을 B 계정에서 바꾸거나 지우려 해도 데이터가 반환되지 않고 원래 할 일도 그대로인지 함께 확인해요.
- 에이전트가 알려 준 배포 준비물을 자기 노트에 남겼는지, 앱이 내 컴퓨터에서 계속 정상적으로 실행되는지 확인해요.
- 배포까지 도전했다면 백엔드 접속 URL과 정상 작동을 확인하는 방법을 에이전트에게 물어보고, 안내대로 확인해요.

**로컬 vs 클라우드, 언제 뭘 쓸까요?** 로컬 DB는 개발·학습·오프라인 작업에, 클라우드 DB는 인터넷을 통해 공동 저장소에 접근해야 할 때 쓸 수 있어요. DB의 저장 위치를 바꾸는 것과 앱을 인터넷에 공개하는 것은 별개예요.

방금 안내받은 배포 준비물은 이 Todo 앱을 다른 사람도 인터넷으로 사용할 수 있게 백엔드를 클라우드에 올릴 때 필요한 것들이에요. 백엔드를 내 컴퓨터에서만 실행하는 대신 Render처럼 서버 코드를 실행해 주는 클라우드 서비스에 올려 두면, 인터넷으로 들어오는 요청을 받을 수 있어요.

> **[과제] 과제 제출 - 내 Todo 앱과 PRD**
>
> - **무엇을** 길 A를 완료한 뒤 길 B로 전환한 Todo 앱과 1부에서 작성한 `PRD.md`를 제출해요.
> - **무엇으로** 작동 화면(또는 3분 시연) + `PRD.md` 한 장. 클라우드 전환과 RLS 확인 기록을 한 줄씩 남겨 주세요. 배포에 도전해 성공했다면 접속 URL도 함께 남겨도 좋아요.
> - **확인** 로컬과 Supabase에서 각각 앱을 껐다 켜도 할 일이 남는지, 삭제·완료가 동작하는지 확인한 장면을 포함해요.
>
> 마감: 2026-11-05T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | PRD 한 장 | 만들 기능과 조건이 담긴 PRD.md가 제출물에 있다. | 30 |
> | 데이터가 남는 Todo 앱 | CRUD가 동작하고 앱을 껐다 켜도 데이터가 남는 장면이 있다. | 40 |
> | 로컬·클라우드 저장과 보안 확인 | 로컬과 Supabase에서 각각 데이터 저장을 확인한 결과와 클라우드 전환·RLS 확인 기록이 있어요. | 30 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

### 오늘의 회고

오늘의 회고를 자기 노트에 짧게 남겨 봐요.

- 새로고침해도 남아야 할 데이터는 무엇이며, 백엔드와 DB는 각각 어떤 일을 하나요?(sec_1～3)
- 데이터베이스가 파일에서 관계형·NoSQL·클라우드로 발전한 이유를 하나 골라 설명해 보세요.(sec_4)
- 내 Todo 앱의 표·항목·관계는 어떻게 정했나요? `SELECT … FROM … WHERE …`를 내 데이터로 풀어 보세요.(sec_5)
- 로그인한 사용자가 자기 할 일만 보게 하려면 어떤 확인이 필요한가요?(sec_6)
- SQL 주입과 비밀번호 유출을 막고, 데이터를 복구하기 위해 AI에게 무엇을 요구할까요?(sec_7)
- 고른 DB의 무료 한도와 백업 정책은 무엇인가요?(sec_8)
- 로컬과 Supabase에서 각각 할 일을 추가하고 새로고침했을 때 남았나요? 확인한 결과를 기록해 보세요.(sec_9)

### 출처 · 더 찾아보기

- **처음 배우는 사람을 위한 데이터베이스란**: Stackby, "What is a database: A beginner's guide." https://stackby.com/blog/what-is-a-database/
- **데이터베이스의 역사**: Cockroach Labs, "A brief history of databases." https://www.cockroachlabs.com/blog/history-of-databases-distributed-sql/
- **NoSQL 종류**: MongoDB, "What Is NoSQL? NoSQL Databases Explained." https://www.mongodb.com/resources/basics/databases/nosql-explained
- **SQL 주입과 방어**: OWASP, "SQL Injection Prevention Cheat Sheet." https://cheatsheetseries.owasp.org/cheatsheets/SQL_Injection_Prevention_Cheat_Sheet.html
- **Little Bobby Tables(xkcd)**: "Exploits of a Mom." https://xkcd.com/327/
- **SQL 주입 막는 법(언어별 예시)**: bobby-tables.com. https://bobby-tables.com/
- **데이터 손실 참사와 백업 교훈**: EnvisionIT, "Six Horror Stories of Data Loss." https://blog.envisionitsolutions.com/six-horror-stories-of-data-loss
- **무료 데이터베이스 비교(2026)**: agentdeals, "Database Free Tier Comparison 2026." https://agentdeals.dev/database-free-tier-comparison-2026
- **LHC 충돌 수**: CERN, “Facts and figures about the LHC.” https://home.cern/resources/faqs/facts-and-figures-about-lhc (2026-02 확인)
- **트랜잭션 격리**: PostgreSQL, “Transaction Isolation.” https://www.postgresql.org/docs/current/transaction-iso.html
- **무료 정책 공식 페이지**: Supabase / Neon / Turso / PlanetScale 가격 페이지(위 무료 DB 비교 셀 링크, 2026-02 확인)

### 핵심 정리

| 섹션 | 오늘 배운 것 | 한 문장 정리 | 내가 할 수 있는 행동 |
|---|---|---|---|
| sec_1 | 오늘의 목표 | 새로고침해도 데이터가 남는 서비스를 만듭니다 | 남겨야 할 데이터를 고릅니다 |
| sec_2 | 백엔드의 역할 | 요청을 처리하고 규칙을 검사해 응답합니다 | 필요한 API와 CRUD 동작을 말합니다 |
| sec_3 | 데이터베이스의 역할 | 데이터를 남기고 필요한 것을 찾아 줍니다 | 화면의 메모리와 DB 저장을 구분합니다 |
| sec_4 | 역사와 갈래 | 파일·관계형·NoSQL·클라우드는 서로 다른 문제를 풀며 등장했습니다 | 내 데이터에 맞는 저장 방식을 비교합니다 |
| sec_5 | SQL과 데이터의 모양 | 표·항목·관계를 정하고 SQL로 데이터를 다룹니다 | 설계도를 그리고 조회 조건을 말합니다 |
| sec_6 | 회원가입과 로그인 | 회원 정보와 로그인 상태도 안전하게 다룰 데이터입니다 | 검증된 인증 도구와 사용자별 접근 제한을 요청합니다 |
| sec_7 | 보안·백업 | 파라미터화·비밀번호 해시·백업과 복구 확인으로 데이터를 지킵니다 | AI에게 방어 방법을 요구하고 복구 여부를 확인합니다 |
| sec_8 | 무료 DB 고르기 | 무료 한도와 백업 정책은 서비스마다 다릅니다 | 가입 직전 공식 정책을 확인합니다 |
| sec_9 | 실습·과제 | Todo 앱을 로컬 DB와 Supabase에 각각 연결합니다 | 저장된 행과 새로고침 뒤의 화면을 확인합니다 |